In [2]:
import pandas as pd

path = "Anonymized MSI Graduates W26 + W27.xlsx - Report 1.csv"
df_raw = pd.read_csv(path, dtype=str)
print(f"Raw shape: {df_raw.shape}")
print(f"\nAdmit terms in file:")
print(df_raw.groupby("Admit Term Descrshort")["Emplid"].nunique())

Raw shape: (7450, 15)

Admit terms in file:
Admit Term Descrshort
FA 2024    284
FA 2025    306
Name: Emplid, dtype: int64


**Data Cleaning**

In [3]:
# Clean column names
df_raw.columns = (
    df_raw.columns.str.strip()
    .str.replace(r"[.\s]+", "_", regex=True)
    .str.replace("-", "", regex=False)
)

In [4]:
# Build derived columns
df_raw["Subject"]      = df_raw["Subject"].str.strip()
df_raw["Catalog_Nbr"]  = df_raw["Catalog_Nbr"].str.strip()
df_raw["Course_Code"]  = df_raw["Subject"] + " " + df_raw["Catalog_Nbr"]
df_raw["Outside_UMSI"] = df_raw["Subject"].apply(lambda x: "No" if x == "SI" else "Yes")
df_raw["Emplid"]       = df_raw["Emplid"].str.strip()

In [5]:
# Filter to FA 2024 cohort only
df = df_raw[df_raw["Admit_Term_Descrshort"] == "FA 2024"].copy()

all_students = set(df["Emplid"].unique())
print(f"Number of Students: {len(all_students)}")
print(f"Number of Unique Courses: {df['Course_Code'].nunique()}")
print(f"\nMost Popular Courses:")
print(df["Course_Code"].value_counts().head(10))
print(f"\nAcad Plan distribution:")
print(df.groupby("Acad_Plan_Descr")["Emplid"].nunique().sort_values(ascending=False))

Number of Students: 284
Number of Unique Courses: 266

Most Popular Courses:
Course_Code
SI 500    279
SI 699    254
SI 506    210
SI 511    171
SI 681    163
SI 539    153
SI 588    129
SI 622    128
SI 582    128
SI 690    126
Name: count, dtype: int64

Acad Plan distribution:
Acad_Plan_Descr
User Experience Design MSI        136
Big Data Analytics MSI             79
Lib, Arch&Knwl Envs in Soc MSI     28
User-Centered Agile Dev MSI        25
Master's ThesisOption Prog MSI     15
Business Administration MBA         2
Information MSI                     1
Name: Emplid, dtype: int64


**Track Assignment & Filtering**

Dual-degree students appear under multiple academic plans in the data. We assign each student to their MSI track (priority 1), treating MBA/MSW plans as secondary. This ensures dual-degree students like the MBA+UX student (77263385) are correctly included in the UX analysis, with their MBA coursework counting as outside-UMSI electives.

In [6]:
# Priority-based plan assignment
# MSI plans (priority 1) always win over MBA/MSW (priority 2)
plan_priority = {
    "User Experience Design MSI":     ("UX",    1),
    "Big Data Analytics MSI":         ("BDA",   1),
    "User-Centered Agile Dev MSI":    ("UCAD",  1),
    "Lib, Arch&Knwl Envs in Soc MSI": ("LAKES", 1),
    "Information MSI":                ("LAKES", 1),  # legacy plan name
    "Master\'s ThesisOption Prog MSI": ("MTOP",  1),
    "Business Administration MBA":    ("MBA",   2),  # secondary
    "Int Prac IH MH&Subst Abuse MSW": ("MSW",   2),  # secondary
}

In [7]:
student_tracks = {}
for emplid, group in df.groupby("Emplid"):
    best_track, best_priority = None, 99
    for plan in group["Acad_Plan_Descr"].unique():
        if plan in plan_priority:
            track, priority = plan_priority[plan]
            if priority < best_priority:
                best_track, best_priority = track, priority
    student_tracks[emplid] = best_track

df["Track"] = df["Emplid"].map(student_tracks)

In [8]:
# Report dual-degree students
dual_degree = [
    eid for eid, group in df.groupby("Emplid")
    if group["Acad_Plan_Descr"].nunique() > 1
]
print(f"Dual-degree students detected: {len(dual_degree)}")

Dual-degree students detected: 2


In [9]:
for eid in dual_degree:
    plans = df[df["Emplid"]==eid]["Acad_Plan_Descr"].unique().tolist()
    assigned = student_tracks[eid]
    print(f"  {eid}: {plans} → assigned to {assigned}")

  77263385: ['Business Administration MBA', 'User Experience Design MSI'] → assigned to UX
  97902838: ['Business Administration MBA', 'User Experience Design MSI'] → assigned to UX


In [10]:
si699_students = set(df.loc[df["Course_Code"] == "SI 699", "Emplid"])
print("SI 699:", len(si699_students))

SI 699: 254


In [11]:
# Drop students with only non-MSI plans (no MSI track at all)
non_msi_ids = {eid for eid, t in student_tracks.items() if t in ("MBA", "MSW", None)}
print(f"\nDropping {len(non_msi_ids)} student(s) with only non-MSI plans")
df = df[~df["Emplid"].isin(non_msi_ids)].copy()


Dropping 0 student(s) with only non-MSI plans


In [12]:
mtop_students = set(df.loc[df["Course_Code"].isin(["SI 697", "SI 698"]), "Emplid"])
print("MTOP:", mtop_students, len(mtop_students))

MTOP: {'76520398', '10834434', '57187132', '59389368', '59931244', '64213632', '17546516', '30005912', '47800917', '64848558', '82334218', '88162771', '67129465', '87636480', '19466574'} 15


In [13]:
# Drop MTOP
mtop_count = (df.groupby("Emplid")["Track"].first() == "MTOP").sum()
print(f"Dropping {mtop_count} MTOP student(s)")
df = df[df["Track"] != "MTOP"].copy()

Dropping 15 MTOP student(s)


In [14]:
missing_students = all_students - (si699_students | mtop_students)
print("Missing Students:", missing_students, len(missing_students))

Missing Students: {'68296282', '09193480', '05130600', '05867384', '53563350', '63146588', '56251501', '63920218', '29149907', '96032491', '85116161', '08138060', '97552274', '62588177', '84087460', '90200100'} 16


In [15]:
# Keep only students who took SI 699
si699_ids = set(df.loc[df["Course_Code"] == "SI 699", "Emplid"])
missing = set(df["Emplid"]) - si699_ids
print(f"Dropping {len(missing)} student(s) without SI 699")

Dropping 16 student(s) without SI 699


In [16]:
df_filtered = df[df["Emplid"].isin(si699_ids)].copy()
print(f"\nStudents for analysis: {df_filtered['Emplid'].nunique()}")
print(f"\nFinal track distribution:")
print(df_filtered.groupby("Track")["Emplid"].nunique().sort_values(ascending=False))


Students for analysis: 253

Final track distribution:
Track
UX       126
BDA       76
LAKES     27
UCAD      24
Name: Emplid, dtype: int64


**Required Course Sets — FA 2024 Cohort PDFs**

Key changes from FA 2023:
- **Core**: SI 500 replaces SI 501
- **UCAD**: Pick-1 selective expanded to include 9 design/UX courses
- **LAKES**: Unified single pathway; SI 581 and SI 564 are now Pick-4 selectives
- **BDA**: SI 561 removed from selectives

In [17]:
UNIVERSAL_EXCLUDE = {
    "SI 500",  # core (replaces SI 501)
    "SI 501",  # legacy core
    "SI 505",  # bridge course
    "SI 506",  # core — Programming I
    "SI 681",  # internship
    "SI 690",  # internship
    "SI 699",  # mastery/capstone
}


In [18]:
REQUIRED = {
    "BDA": {
        # Mastery Prereq fixed
        "SI 504","SI 507","SI 544","SI 568","SI 602","SI 618",
        # Pick 1 MP
        "SI 670","SI 671",
        # Pick 2 selectives
        "SI 608","SI 630","SI 649","SI 650",
    },
    "UX": {
        # Mastery Prereq fixed
        "SI 520","SI 539","SI 582","SI 588","SI 622",
        # Pick 2 selectives
        "SI 529","SI 552","SI 559","SI 612","SI 616","SI 658","SI 659","SI 684",
    },
    "UCAD": {
        # Mastery Prereq fixed
        "SI 504","SI 539","SI 579","SI 582","SI 588","SI 622",
        # Pick 1 MP
        "SI 664","SI 669",
        # Pick 1 Selective (expanded in FA 2024)
        "SI 520","SI 529","SI 552","SI 559","SI 612","SI 616","SI 658","SI 659","SI 684",
    },
    "LAKES": {
        # Mastery Prereq fixed
        "SI 510","SI 580","SI 647","SI 666",
        # Pick 1 MP
        "SI 667","SI 547",
        # Pick 4 selectives
        "SI 564","SI 639","SI 581","SI 583","SI 585","SI 623","SI 626",
        "SI 632","SI 633","SI 643","SI 676",
    },
}

**SI Elective Analysis**

In [19]:
results = {}
for track in ["BDA", "UX", "UCAD", "LAKES"]:
    exclude = UNIVERSAL_EXCLUDE | REQUIRED[track]
    track_ids = df_filtered[df_filtered["Track"] == track]
    n_students = track_ids["Emplid"].nunique()

    track_df = track_ids[
        (~track_ids["Course_Code"].isin(exclude)) &
        (track_ids["Outside_UMSI"] == "No")
    ]

    counts = (
        track_df.groupby("Course_Code")["Emplid"]
        .nunique().sort_values(ascending=False)
        .rename("Student_Count").reset_index()
    )
    counts["Pct_of_Track"] = (counts["Student_Count"] / n_students * 100).round(1)
    results[track] = counts

    print(f"\n\u2500\u2500 {track} (n={n_students} students) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
    print(counts.head(10).to_string(index=False))


── BDA (n=76 students) ──────────────────
Course_Code  Student_Count  Pct_of_Track
     SI 564             46          60.5
     SI 664             42          55.3
     SI 644             22          28.9
     SI 511             18          23.7
     SI 539             18          23.7
     SI 563             18          23.7
     SI 639              9          11.8
     SI 579              7           9.2
     SI 643              7           9.2
     SI 631              6           7.9

── UX (n=126 students) ──────────────────
Course_Code  Student_Count  Pct_of_Track
     SI 511             88          69.8
     SI 611             62          49.2
     SI 594             39          31.0
     SI 504             23          18.3
     SI 538             17          13.5
     SI 534             15          11.9
     SI 579             14          11.1
     SI 564             12           9.5
     SI 691             12           9.5
     SI 540              9           7.1

── UCAD (n=

**Outside UMSI Elective Analysis**

Note: For dual-degree students, their non-SI program courses (e.g. MBA courses) are included here as outside-UMSI electives.

In [20]:
outside_results = {}
for track in ["BDA", "UX", "UCAD", "LAKES"]:
    exclude = UNIVERSAL_EXCLUDE | REQUIRED[track]
    track_ids = df_filtered[df_filtered["Track"] == track]
    n_students = track_ids["Emplid"].nunique()

    outside_df = track_ids[
        (~track_ids["Course_Code"].isin(exclude)) &
        (track_ids["Outside_UMSI"] == "Yes")
    ]

    counts = (
        outside_df.groupby("Course_Code")["Emplid"]
        .nunique().sort_values(ascending=False)
        .rename("Student_Count").reset_index()
    )
    counts["Pct_of_Track"] = (counts["Student_Count"] / n_students * 100).round(1)
    outside_results[track] = counts

    print(f"\n\u2500\u2500 {track} (n={n_students} students) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
    print(counts.head(11).to_string(index=False))


── BDA (n=76 students) ──────────────────
 Course_Code  Student_Count  Pct_of_Track
    ENTR 500              9          11.8
    EECS 553              5           6.6
     EAS 506              4           5.3
      TO 512              4           5.3
    ENTR 550              4           5.3
      TO 638              3           3.9
    EECS 545              3           3.9
      TO 572              2           2.6
ARTSADMN 533              2           2.6
   STATS 503              2           2.6
     ELI 510              2           2.6



── UX (n=126 students) ──────────────────
Course_Code  Student_Count  Pct_of_Track
   ENTR 560             16          12.7
   ENTR 500             12           9.5
   ENTR 550             10           7.9
    MKT 613              8           6.3
   ENTR 599              7           5.6
   ENGR 599              6           4.8
  UARTS 560              6           4.8
     MO 512              5           4.0
     TO 512              4           3.2
    MKT 618              4           3.2
 SIABRD 504              3           2.4

── UCAD (n=24 students) ──────────────────
Course_Code  Student_Count  Pct_of_Track
   EDUC 546              2           8.3
   ENGR 599              2           8.3
   ENTR 500              2           8.3
   ENTR 560              2           8.3
   ECON 101              1           4.2
   EDUC 602              1           4.2
   EDUC 616              1           4.2
   EDUC 620              1           4.2
   EECS 498              1           4.2
   EECS 542

In [21]:
# Export full elective list for all tracks — for student-facing course directory
import json

full_course_map = {}

for track, df_result in results.items():
    # Only include SI courses (not outside UMSI)
    si_only = df_result[df_result['Course_Code'].str.startswith('SI ')]
    for _, row in si_only.iterrows():
        code = row['Course_Code']
        name = row.get('Crse_Descr') or row.get('Crse Descr') or ''
        if code not in full_course_map:
            full_course_map[code] = {'name': name, 'tracks': []}
        if track not in full_course_map[code]['tracks']:
            full_course_map[code]['tracks'].append(track)

# Print as JavaScript array ready to paste into your HTML file
print('const allCourses = [')
for code in sorted(full_course_map.keys(), key=lambda x: int(x.split()[1])):
    info = full_course_map[code]
    tracks = sorted([t.lower() for t in info['tracks']])
    tracks_str = ', '.join(f"'{t}'" for t in tracks)
    name = info['name'].replace("'", "\\'")
    print(f"  {{ code: '{code}', name: '{name}', tracks: [{tracks_str}] }},")
print('];')

const allCourses = [
  { code: 'SI 504', name: '', tracks: ['lakes', 'ux'] },
  { code: 'SI 507', name: '', tracks: ['ucad', 'ux'] },
  { code: 'SI 510', name: '', tracks: ['bda', 'ux'] },
  { code: 'SI 511', name: '', tracks: ['bda', 'lakes', 'ucad', 'ux'] },
  { code: 'SI 512', name: '', tracks: ['lakes', 'ucad', 'ux'] },
  { code: 'SI 515', name: '', tracks: ['ucad', 'ux'] },
  { code: 'SI 519', name: '', tracks: ['bda', 'lakes', 'ucad', 'ux'] },
  { code: 'SI 529', name: '', tracks: ['bda', 'lakes'] },
  { code: 'SI 534', name: '', tracks: ['bda', 'ucad', 'ux'] },
  { code: 'SI 538', name: '', tracks: ['ux'] },
  { code: 'SI 539', name: '', tracks: ['bda', 'lakes'] },
  { code: 'SI 540', name: '', tracks: ['bda', 'lakes', 'ucad', 'ux'] },
  { code: 'SI 542', name: '', tracks: ['bda', 'ucad', 'ux'] },
  { code: 'SI 544', name: '', tracks: ['lakes', 'ucad', 'ux'] },
  { code: 'SI 547', name: '', tracks: ['ux'] },
  { code: 'SI 548', name: '', tracks: ['ucad', 'ux'] },
  { code: 'SI 5